<a href="https://colab.research.google.com/github/goitstudent123/llm-gen/blob/main/dz_topic_6_has.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Chunk 1: Clean install compatible LangChain packages
!pip -q uninstall -y langchain langchain-core langchain-openai langchain-community langchain-text-splitters langgraph langgraph-checkpoint langgraph-prebuilt langgraph-sdk openai

!pip -q install --no-cache-dir -U \
    "langchain>=1.0.0" \
    "langchain-openai>=1.0.0" \
    "langgraph>=1.0.0" \
    "openai>=1.0.0"

!pip -q check

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.7/96.7 kB 81.0 MB/s eta 0:00:00


In [2]:
# Chunk 1.1: Verify installed package versions
import importlib.metadata as metadata

packages = [
    "langchain",
    "langchain-core",
    "langchain-openai",
    "langgraph",
    "openai",
]

for package_name in packages:
    print(f"{package_name}: {metadata.version(package_name)}")

langchain: 1.2.17
langchain-core: 1.3.2
langchain-openai: 1.2.1
langgraph: 1.1.10
openai: 2.33.0


In [3]:
# Chunk 2: Initial imports and OpenRouter config
import os
import uuid
from datetime import datetime, timezone

from IPython.display import Markdown, display

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore
from langgraph.config import get_store


def read_colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

openrouter_api_key = (
    read_colab_secret("OPENROUTER_API_KEY")
    or os.getenv("OPENROUTER_API_KEY")
)

if not openrouter_api_key:
    raise ValueError(
        "Add OPENROUTER_API_KEY to Colab Secrets or set it in os.environ."
    )

OPENROUTER_MODEL = (
    read_colab_secret("OPENROUTER_MODEL")
    or os.getenv("OPENROUTER_MODEL")
    or "openai/gpt-oss-120b:nitro"
)

OPENROUTER_HTTP_REFERER = (
    read_colab_secret("OPENROUTER_HTTP_REFERER")
    or os.getenv("OPENROUTER_HTTP_REFERER")
    or "https://colab.research.google.com"
)

OPENROUTER_APP_TITLE = (
    read_colab_secret("OPENROUTER_APP_TITLE")
    or os.getenv("OPENROUTER_APP_TITLE")
    or "Colab TODO Agent"
)

In [4]:
# Chunk 3: Utilities and shared constants
TASK_NAMESPACE = ("todo_tasks",)
TASK_LIMIT = 1000


def normalize_title(title: str) -> str:
    return " ".join(title.strip().split())


def short_task_id(task_id: str) -> str:
    return task_id[:8]


def format_task_line(index: int, task_id: str, title: str) -> str:
    return f"{index}. {title} [{short_task_id(task_id)}]"


def find_task_by_id_or_prefix(store, task_id: str):
    clean_task_id = task_id.strip()

    item = store.get(TASK_NAMESPACE, clean_task_id)
    if item is not None:
        return item.key, item

    matches = [
        item
        for item in store.search(TASK_NAMESPACE, limit=TASK_LIMIT)
        if item.key.startswith(clean_task_id)
    ]

    if len(matches) == 1:
        return matches[0].key, matches[0]

    return None, None

In [5]:
# Chunk 4: TODO tools that use long-term store
@tool
def add_task(title: str) -> str:
    """Add a new TODO task with the given title."""
    clean_title = normalize_title(title)

    if not clean_title:
        return "Не можу додати порожнє завдання."

    store = get_store()
    task_id = str(uuid.uuid4())

    store.put(
        TASK_NAMESPACE,
        task_id,
        {
            "id": task_id,
            "title": clean_title,
            "created_at": datetime.now(timezone.utc).isoformat(),
        },
    )

    return f'Завдання "{clean_title}" було успішно додано. ID: {short_task_id(task_id)}'


@tool
def list_tasks() -> str:
    """List all saved TODO tasks."""
    store = get_store()
    items = store.search(TASK_NAMESPACE, limit=TASK_LIMIT)

    if not items:
        return "Список завдань порожній."

    sorted_items = sorted(
        items,
        key=lambda item: item.value.get("created_at", ""),
    )

    lines = ["Ось всі ваші завдання:"]

    for index, item in enumerate(sorted_items, 1):
        title = item.value.get("title", "Без назви")
        lines.append(format_task_line(index, item.key, title))

    return "\n".join(lines)


@tool
def delete_task(task_id: str) -> str:
    """Delete a TODO task by its full UUID or visible UUID prefix."""
    store = get_store()
    real_task_id, item = find_task_by_id_or_prefix(store, task_id)

    if item is None:
        return f'Завдання з ID "{task_id}" не знайдено.'

    title = item.value.get("title", "Без назви")
    store.delete(TASK_NAMESPACE, real_task_id)

    return f'Завдання "{title}" було видалено.'

In [6]:
# Chunk 5: Create TODO agent with InMemoryStore and OpenRouter model
store = InMemoryStore()

model = ChatOpenAI(
    model=OPENROUTER_MODEL,
    api_key=openrouter_api_key,
    base_url=OPENROUTER_BASE_URL,
    temperature=0,
    default_headers={
        "HTTP-Referer": OPENROUTER_HTTP_REFERER,
        "X-OpenRouter-Title": OPENROUTER_APP_TITLE,
    },
)

SYSTEM_PROMPT = """
You are a Ukrainian-speaking TODO assistant.

You manage tasks using only the provided tools:
- add_task
- list_tasks
- delete_task

Rules:
- Understand Ukrainian user commands.
- For add requests, extract the task title and call add_task.
- For list or remaining-tasks requests, call list_tasks.
- For delete requests with an exact ID, call delete_task directly.
- For delete requests that mention a topic or title instead of an ID, first call list_tasks, find the matching task, then call delete_task with the shown ID prefix.
- Do not invent task IDs.
- Keep final answers short and natural in Ukrainian.
- Do not mention internal tool calls.
"""

agent = create_agent(
    model=model,
    tools=[add_task, list_tasks, delete_task],
    store=store,
    system_prompt=SYSTEM_PROMPT,
)

In [7]:
# Chunk 6: Helper for extracting the final agent answer
def get_last_message_text(result: dict) -> str:
    messages = result.get("messages", [])

    for message in reversed(messages):
        content = getattr(message, "content", None)

        if isinstance(content, str) and content.strip():
            return content.strip()

        if content:
            return str(content)

    return str(result)

In [8]:
# Chunk 7: Run acceptance tests
tests = [
    "Додай: купити хліб",
    "Додай: подзвонити лікарю",
    "Покажи всі завдання",
    "Видали завдання про хліб",
    "Що залишилось?",
]


for i, test in enumerate(tests, 1):
    display(Markdown(f"## TEST {i}: {test}"))

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": test,
                }
            ]
        }
    )

    answer = get_last_message_text(result)
    print(answer)

## TEST 1: Додай: купити хліб

Завдання додано. Якщо потрібно ще щось – дайте знати!


## TEST 2: Додай: подзвонити лікарю

Завдання додано. Якщо потрібно переглянути список або щось інше – дайте знати.


## TEST 3: Покажи всі завдання

Осьваш список завдань:

1. купити хліб [3f458bf7]  
2. подзвонити лікарю [f811a28b]


## TEST 4: Видали завдання про хліб

Завдання про хліб видалено.


## TEST 5: Що залишилось?

Залишилосяодне завдання:

- подзвонити лікарю [f811a28b]
